> 📄 English: [agentcore_tutorial.ipynb](agentcore_tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--aws-agentcore--agentcore-tutorial)

# AWS Bedrock AgentCore 教程：从本地 Agent 到托管 Agent

## 概览

本教程介绍 AWS Bedrock AgentCore，这是一个运行时框架，旨在把本地 AI Agent 转变为生产就绪的系统。我们将带你走完创建一个简单 Agent，然后用 AgentCore 包装它、以实现托管执行、请求跟踪和标准化通信模式的全过程。

学完本教程，你将理解 AgentCore 如何充当位于你的 Agent 逻辑与生产环境之间的一层基础设施，处理请求管理、执行控制和运行时编排等复杂问题。

### 你将学到什么

本教程涵盖三个基本概念。首先，你将学习如何使用标准 Python 构建一个基础 Agent。其次，你将理解如何使用 Python SDK 用 AgentCore 运行时包装你的 Agent。最后，你将看到如何通过一个标准化的 HTTP 接口在本地运行你的 Agent，该接口提供请求跟踪和结构化响应。


## 理解 AgentCore 架构

在动手实现之前，先理解 AgentCore 做什么以及为什么重要会很有帮助。当你在本地构建一个 Agent 时，它通常只是一个接收输入并返回输出的 Python 函数或类。这在开发和测试时没问题，但生产环境需要额外的基础设施。

AgentCore 提供了这层基础设施。可以把它想象成你的 Agent 的“控制塔”。控制塔不驾驶飞机，但它管理起飞、降落和通信。类似地，AgentCore 不改变你的 Agent 逻辑，但它管理请求如何到达你的 Agent、响应如何返回，以及整个执行过程如何被跟踪和监控。

该框架引入了几个关键组件。运行时服务器负责处理传入请求并将其路由到你的 Agent。SDK 提供了一个简单的装饰器模式，把你的函数转换为一个 HTTP 服务。这些组件共同创建了一个托管执行环境，将本地函数转变为生产服务。

### 为什么使用 AgentCore

AgentCore 的主要价值在于关注点分离。你的 Agent 代码只专注于处理消息和生成响应。AgentCore 处理其余所有事情：接收 HTTP 请求、校验输入、跟踪执行时间、管理错误以及格式化响应。这种分离让你的代码更干净、更易测试、更易维护。

此外，AgentCore 为生产能力提供了基础。一旦你的 Agent 通过运行时运行，你就可以添加工具、启用日志、实现访问控制，并部署到任意服务器，而无需修改你的核心 Agent 逻辑。基础设施是在你的 Agent 周围生长，而不是长在它内部。


## 系统架构图

下图展示了 AgentCore 如何把本地 Agent 转变为托管系统。流程从客户端发起的 HTTP 请求开始，经过 AgentCore 运行时，到达你的 Agent 进行处理，然后沿着同样的路径返回并附带额外的元数据。


![AgentCore Architecture Diagram](assets/agentcore-diagram.png)


## 第 1 步：环境配置

我们首先安装必要的依赖并配置环境。AgentCore 需要 Python 3.10 或更高版本，并依赖若干核心库用于 HTTP 处理和配置管理。


### 安装依赖

以下命令会安装 Bedrock AgentCore Runtime SDK 以及配套库。`python-dotenv` 包帮助我们安全管理 API key，而 `requests` 让我们可以通过发起 HTTP 调用来测试我们的 Agent。


In [ ]:
!pip install bedrock-agentcore python-dotenv requests openai

### 配置 API 凭据

在本教程中，我们使用 OpenAI 的 API 来驱动我们 Agent 的语言理解能力。以下代码从 `.env` 文件加载你的 API key，这是避免把敏感凭据写进代码的最佳实践。你应该在工作目录下创建一个 `.env` 文件，其中包含类似 `OPENAI_API_KEY=your-key-here` 这样一行。


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## 第 2 步：构建基础 Agent

现在我们来创建我们的 Agent。这是一个标准的 Python 函数，用于处理消息并返回响应。为了让重点放在理解 AgentCore 如何包装和管理它上，这个 Agent 有意保持简单。


### Agent 实现

`process_message` 函数演示了一个使用 OpenAI API 回答问题的基础 Agent。这只是普通 Python 代码，没有混入任何 AgentCore 特有的逻辑。请注意，我们把 Agent 逻辑定义为一个接收消息并返回响应的简单函数。


In [ ]:
import openai

def process_message(message: str) -> str:
    """Process a message and return a response using OpenAI"""
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": message}],
        temperature=0
    )
    return response.choices[0].message.content

### 测试本地 Agent

在集成 AgentCore 之前，我们先在隔离状态下验证 Agent 能否正确工作。这一步很重要，因为它能确认之后遇到的任何问题都来自运行时集成，而不是 Agent 逻辑本身。


In [ ]:
result = process_message("What is AI?")
print("Agent Response:")
print(result)

你应该会看到一段清晰、信息丰富的人工智能说明。此时，我们已经有了一个可以工作的 Agent，但它只是一个本地 Python 函数。没有服务器、没有请求处理，外部系统也无法与它交互。这就是 AgentCore 登场的地方。


## 第 3 步：集成 AgentCore 运行时

这是转换步骤，我们要用 AgentCore 的运行时基础设施来包装本地 Agent。这个过程包括创建一个 BedrockAgentCoreApp 实例、用装饰器标记我们的入口函数，以及理解服务器的工作原理。每一部分都在创建托管执行环境时扮演特定角色。


### 创建 AgentCore 应用

`BedrockAgentCoreApp` 类是 SDK 的核心。当你创建实例时，它会搭建处理 HTTP 请求、健康检查和响应格式化所需的所有基础设施。这一行代码就初始化了整个运行时框架。


In [ ]:
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

print("Bedrock AgentCore application created")

### Agent 注册

注册是我们把 Agent 代码连接到运行时的方式。`@app.entrypoint` 装饰器告诉 AgentCore，每当有请求到达时就调用我们的函数。函数签名很重要：它接收一个包含请求数据的 `payload` 字典。按照约定，AgentCore 期望用户消息放在名为 `prompt` 的字段中。

请注意，在被装饰的函数内部，我们调用了已有的 `process_message` 函数。这种模式让你在创建和管理 Agent 时拥有灵活性。你可以在这里使用依赖注入、实现缓存或添加初始化逻辑，而无需改动 AgentCore 集成。


In [ ]:
@app.entrypoint
def invoke(payload: dict) -> dict:
    """
    Process user input and return a response.
    AgentCore will call this function for each HTTP request to /invocations
    """
    user_message = payload.get("prompt", "Hello")
    result = process_message(user_message)
    return {"result": result}

print("Agent registered with AgentCore")

### 启动运行时服务器

现在我们演示如何启动将接收 HTTP 请求的实际服务器。`app.run()` 方法默认在 8080 端口启动一个 HTTP 服务器。在实际部署中，你会把这段代码保存到一个文件（例如 `agent.py`）中，然后用 `python agent.py` 运行它。

注意：在 notebook 中直接运行 `app.run()` 会阻塞执行。在生产环境中，你会在单独进程中运行它，或使用 AgentCore starter toolkit 进行部署。


In [ ]:
# In a real scenario, you would run this in a separate file:
# if __name__ == "__main__":
#     app.run()

# For notebook demonstration, we show the complete agent code:
complete_agent_code = '''
import os
import openai
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from dotenv import load_dotenv

load_dotenv()

app = BedrockAgentCoreApp()

def process_message(message: str) -> str:
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": message}],
        temperature=0
    )
    return response.choices[0].message.content

@app.entrypoint
def invoke(payload: dict) -> dict:
    user_message = payload.get("prompt", "Hello")
    result = process_message(user_message)
    return {"result": result}

if __name__ == "__main__":
    app.run()  # Server starts on http://localhost:8080
'''

print("Complete agent code structure:")
print(complete_agent_code)
print("\nKey endpoints when running:")
print("  POST http://localhost:8080/invocations - Send requests to your agent")
print("  GET http://localhost:8080/ping - Health check endpoint")

此时，你的 Agent 不再只是一个 Python 函数。当你把代码作为独立脚本运行时，它就变成了一个服务，可以接收 HTTP 请求、通过你的 Agent 逻辑处理它们，并返回结构化响应。AgentCore SDK 已自动添加了健康检查端点、请求校验和响应格式化。


## 第 4 步：与托管 Agent 交互

当运行时服务器处于活动状态时（在 notebook 之外运行时），我们可以通过 HTTP 请求与我们的 Agent 交互。这代表着从直接函数调用到基于网络通信的根本转变。Agent 逻辑没有改变，但我们访问它的方式发生了变化。


### 发起你的第一次请求

我们使用 `requests` 库向正在运行的 AgentCore 服务器发送一个 HTTP POST 请求。请求发送到 `/invocations` 端点，JSON 载荷中包含一个 `prompt` 字段。这是 AgentCore 期望的标准化格式。运行时会接收该请求、提取载荷、调用我们被装饰的入口函数，并打包响应。


In [ ]:
# This code would work when the server is running in a separate process:
import requests
import json

# Example HTTP request structure:
request_example = {
    "url": "http://localhost:8080/invocations",
    "method": "POST",
    "headers": {"Content-Type": "application/json"},
    "body": {"prompt": "What is AI?"}
}

print("Example HTTP request to AgentCore:")
print(json.dumps(request_example, indent=2))
print("\nExample using curl:")
print("curl -X POST http://localhost:8080/invocations \\")
print("  -H 'Content-Type: application/json' \\")
print("  -d '{\"prompt\": \"What is AI?\"}'")

# If the server were running, you would make the actual request:
# response = requests.post(
#     "http://localhost:8080/invocations",
#     json={"prompt": "What is AI?"}
# )
# result = response.json()
# print(json.dumps(result, indent=2))

### 理解响应结构

你从 AgentCore 收到的响应遵循你在入口函数中定义的结构。在我们的例子中，我们返回 `{"result": result}`，因此 HTTP 响应会包含这个结构。这让构建消费你 Agent 输出的系统变得容易，因为格式是可预测且一致的。


In [ ]:
# Expected response structure based on our entrypoint function:
expected_response = {
    "result": "AI (Artificial Intelligence) refers to computer systems that can perform tasks typically requiring human intelligence..."
}

print("Expected response structure:")
print(json.dumps(expected_response, indent=2))

### 用不同输入进行测试

为了更好地理解系统如何工作，这里用另一个问题再举一个示例请求。每个请求都通过同一个入口函数独立处理。


In [ ]:
# Another example request:
print("Example request for machine learning explanation:")
print("curl -X POST http://localhost:8080/invocations \\")
print("  -H 'Content-Type: application/json' \\")
print("  -d '{\"prompt\": \"Explain machine learning in simple terms\"}'")

# Expected response:
print("\nExpected response:")
print('{"result": "Machine learning is..."}')

## 对比本地执行与托管执行

值得停下来想一想，什么变了，什么没变。你 Agent 的核心逻辑——调用 OpenAI API 并返回响应的 `process_message` 函数——完全保持不变。你并没有为了配合 AgentCore 而修改 Agent 代码。

变化的是你与 Agent 交互的方式。在 AgentCore 之前，你直接调用函数：

```python
result = process_message("What is AI?")
print(result)
```

在 AgentCore 之后，你发送一个 HTTP 请求：

```python
response = requests.post(
    "http://localhost:8080/invocations",
    json={"prompt": "What is AI?"}
)
result = response.json()
```

这一变化带来了显著的好处。你的 Agent 现在可以通过网络访问，也就是说任何能说 HTTP 的应用都可以调用它。你可以使用 AgentCore starter toolkit 将它部署到 AWS 基础设施上。你可以添加记忆能力、实现访问控制，并扩展以应对生产流量。最重要的是，你已经打下了一个基础，能在不修改核心 Agent 逻辑的前提下支持所有这些增强。

这种关注点分离——Agent 逻辑与基础设施——正是 AgentCore 的关键洞见。你的 Agent 专注于做好自己的任务。AgentCore 专注于让这个 Agent 可访问、可管理、生产就绪。


## 总结

在本教程中，你学习了使用 AWS Bedrock AgentCore 的基本模式。你创建了一个简单 Agent，用 `@app.entrypoint` 装饰器将它包装进 AgentCore 运行时 SDK，并理解了如何通过托管 HTTP 接口与它交互。Agent 代码本身保持干净和专注，而 AgentCore 处理了基础设施方面的问题。

这个基础为多种增强方向打开了大门。你可以添加工具，让 Agent 搜索网页、查询数据库或调用外部 API。你可以启用详细日志，精确查看 Agent 在每一步做了什么。你可以实现身份与访问控制，限制谁能使用某些能力。你可以使用 AgentCore starter toolkit 将 Agent 部署到 AWS，在那里处理真实用户请求。

所有这些增强都建立在你在本教程学到的同一模式之上：你的 Agent 专注于核心任务，而 AgentCore 管理它周围的基础设施。这种分离让你的代码更易测试、更易维护，并能在需求变化时更易演进。
